# Experimento
Vamos a analizar los vocabularios generados por el algorito BM25 tanto de la colección 'texto' como de la colección 'resumen'.

## Primero instalamos llama-index para poder leer los índices generados

In [3]:
!pip install -q llama-index

## Extracción de los vocabularios

Los vocabularios generados por el algoritmo BM25 están en archivos llamados 'vectorizer.pkl'

Hay que cargar los archivos y utilizar el método get_vocabulary() para leerlos.

In [12]:
import sys
from pathlib import Path
import pickle

# Add parent directory to Python path
# Assuming your project root is `/work`, add it to the system path
project_root = Path('/work')
sys.path.append(str(project_root))
from src.utils.data_utils import load_documents
from src.utils.index_classes import SparseBM25Vectorizer

# Define index paths for "texto" and "resumen"
index_path_texto = Path(f'/work/data/indexes/OEB_texto_BM25sparseindex')
index_path_resumen = Path(f'/work/data/indexes/OEB_resumen_BM25sparseindex')

# Load persisted vectorizers
with open(index_path_texto / "vectorizer.pkl", "rb") as f:
    vectorizer_texto = pickle.load(f)

with open(index_path_resumen / "vectorizer.pkl", "rb") as f:
    vectorizer_resumen = pickle.load(f)
    # (We won’t use vectorizer_resumen here since we’re reusing vectorizer_texto for retrieval.)

# Get the vocabulary
vocab_texto = vectorizer_texto.get_vocabulary()
vocab_resumen = vectorizer_resumen.get_vocabulary()


## Presentación
Una vez extraídos vamos a ponerlos en una tabla comparativa

In [13]:
# First, extract the words as sets
words_texto = set(vocab_texto.keys())
words_resumen = set(vocab_resumen.keys())

# Combine all unique words alphabetically
all_words = sorted(words_texto.union(words_resumen))

# Print header
print(f"{'Word':<20} | {'In Texto':<10} | {'In Resumen':<10}")
print('-'*50)

# Iterate and print
for word in all_words:
    in_vocab_texto = 'Yes' if word in words_texto else 'No'
    in_vocab_resumen = 'Yes' if word in words_resumen else 'No'
    print(f"{word:<20} | {in_vocab_texto:<10} | {in_vocab_resumen:<10}")

# Finally, print vocabulary sizes
print('\nSummary:')
print(f"Texto size: {len(words_texto)}")
print(f"Resumen size: {len(words_resumen)}")



Word                 | In Texto   | In Resumen
--------------------------------------------------
0                    | Yes        | Yes       
0,80 a 1,00 m        | No         | Yes       
00                   | Yes        | No        
1                    | Yes        | Yes       
1 o 2                | No         | Yes       
1''                  | No         | Yes       
1,10 m               | No         | Yes       
10                   | Yes        | No        
100mm                | Yes        | No        
110                  | Yes        | Yes       
110mm                | No         | Yes       
12                   | Yes        | Yes       
150                  | Yes        | No        
16                   | Yes        | Yes       
160                  | Yes        | No        
160mm                | No         | Yes       
18                   | Yes        | Yes       
2                    | Yes        | Yes       
2"                   | No         | Yes       
2''      

## Comentario
Como se puede observar los vocabularios no coinciden (algo normal). Se observan múltiples 'stopwords' que podrían eliminarse del vocabulario

## Experimento
Vamos a probar a forzar que los vocabularios tanto para 'texto' como para 'resumen' coincidan y, además, eliminando las 'stopwords'.

Para implementar correctamente el método tendremos que volver a crear los índices con el vocabulario restringido dado que es necesario recalcular los pesos (weights) y los vectores (embeddings).

### Paso 1: Crear el vocabulario restringido
Vamos a usar como base las palabras de:
- Los conceptos
- Los nombres de los parámetros
- Los valores de los parámetros

(Observar que este método produce los mismos resultados tanto si se aplica sobre la colección 'texto' como 'resumen')

In [14]:
# Load the pickle files
with open(index_path_texto / "documents.pkl", "rb") as f:
    doc = pickle.load(f)

# Extract the vocabulary (raw, no stopwords yet)
import re

def extract_words(item):
    words = set()
    
    # From 'concept'
    concept_words = re.findall(r'\w+', item.metadata['concept'].lower())
    words.update(concept_words)
    
    # From 'parameters'
    for param in item.metadata['parameters'].values():
        label_words = re.findall(r'\w+', param.get('label', '').lower())
        words.update(label_words)
        
        # Extract value (add full value, not split)
        for value in param.get('values', []):
            value_word = value.get('value', '').strip().lower()
            if value_word:
                words.add(value_word)
                
    return words

# Extract words from all items
all_words = set()
for item in doc:
    all_words.update(extract_words(item))

# Remove repetitions
unique_words = set(all_words)

# Sort them alphabetically
unique_words_sorted = sorted(unique_words)

# Output
print(f"Total unique words: {len(unique_words)}")
print(unique_words_sorted)


Total unique words: 162
['0', '0,80 a 1,00 m', '1', '1 o 2', "1''", '1,10 m', '110', '110mm', '12', '16', '160mm', '18', '2', '2"', "2''", '20', '200mm', '21mm', '220', '24', '29mm', '3', "3 1/2''", '3 <= i < 5 horas', '3"', "3''", '36mm', '4', "4''", '400', '40mm', '48mm', '5', "5''", '50mm', '5t', '6', "6''", '60', '8', '90mm', 'a', 'acción', 'acero', 'adosada', 'alma', 'anchura', 'andén', 'arqueta', 'bajo vías', 'balasto', 'banda', 'blando', 'cable', 'cables', 'canaleta', 'canalizaciones', 'canalización', 'cata', 'circuito', 'cms', 'con', 'con reposición', 'con topo', 'condiciones', 'conexión', 'corrugado', 'cruce', 'cruce de carretera', 'cualquier condición de ejecución', 'cualquier franja horaria', 'cualquier franja horaria excepcional', 'cuatritubo', 'cámara', 'de', 'de cantera', 'de la traza', 'desde', 'diametro', 'diurno', 'diurno excepcional', 'diámetro', 'diámetros', 'doble', 'drenaje', 'duro', 'ejecución', 'elementos', 'embocadura', 'en', 'entre', 'entronque', 'existente', '

In [15]:
# Remove stopwords
# Define stopwords
stopwords = {"para", "de", "y", "ó", "la", "el", "en", "a", "del", "los", "las", "un", "una", "por", "con", "sin", "o"}

# Filter and deduplicate
vocabulary = {word for word in unique_words_sorted if word not in stopwords}

# Sort if you want
vocabulary_sorted = sorted(vocabulary)

print(vocabulary_sorted)
print(f"Total unique words: {len(vocabulary_sorted)}")

['0', '0,80 a 1,00 m', '1', '1 o 2', "1''", '1,10 m', '110', '110mm', '12', '16', '160mm', '18', '2', '2"', "2''", '20', '200mm', '21mm', '220', '24', '29mm', '3', "3 1/2''", '3 <= i < 5 horas', '3"', "3''", '36mm', '4', "4''", '400', '40mm', '48mm', '5', "5''", '50mm', '5t', '6', "6''", '60', '8', '90mm', 'acción', 'acero', 'adosada', 'alma', 'anchura', 'andén', 'arqueta', 'bajo vías', 'balasto', 'banda', 'blando', 'cable', 'cables', 'canaleta', 'canalizaciones', 'canalización', 'cata', 'circuito', 'cms', 'con reposición', 'con topo', 'condiciones', 'conexión', 'corrugado', 'cruce', 'cruce de carretera', 'cualquier condición de ejecución', 'cualquier franja horaria', 'cualquier franja horaria excepcional', 'cuatritubo', 'cámara', 'de cantera', 'de la traza', 'desde', 'diametro', 'diurno', 'diurno excepcional', 'diámetro', 'diámetros', 'doble', 'drenaje', 'duro', 'ejecución', 'elementos', 'embocadura', 'entre', 'entronque', 'existente', 'ferroviarias', 'fábrica', 'galvanizado', 'hasta 

In [16]:
# Build vocabulary dict as CountVectorizer expects
allowed_vocab_list = sorted(list(vocabulary_sorted))
allowed_vocab_dict = {term: idx for idx, term in enumerate(allowed_vocab_list)}

In [25]:
# Save the vocabulary to a picke file
vocab_path = Path('/work/sandbox/BM25_Filtered_Voc')
vocab_file = vocab_path / 'vocab_dict.pkl'

try:
    with open(vocab_file, 'wb') as f:
        pickle.dump(allowed_vocab_dict, f)
    print(f"Saved vocabulary with {len(allowed_vocab_dict)} terms to {vocab_path}")
except Exception as e:
    print(f"Error saving vocabulary to {vocab_path}: {e}")

Saved vocabulary with 153 terms to /work/sandbox/BM25_Filtered_Voc


### Paso 2: Recrear los índices

In [39]:
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse

# load documents
texto_doc_path = Path(f'/work/data/indexes/OEB_texto_BM25sparseindex')
resumen_doc_path = Path(f'/work/data/indexes/OEB_resumen_BM25sparseindex')

with open(texto_doc_path / "documents.pkl", "rb") as f:
    texto_documents = pickle.load(f)

with open(resumen_doc_path / "documents.pkl", "rb") as f:
    resumen_documents = pickle.load(f)

# Build the vectorizer with restricted vocabulary
vectorizer_restricted = CountVectorizer(vocabulary=allowed_vocab_dict, 
                                        binary=False, 
                                        lowercase=True, 
                                        token_pattern=r'\b\w+\b')

# Create a new SparseBM25Vectorizer instance for texto documents
bm25_restricted_texto = SparseBM25Vectorizer(k1=1.5, b=0.75)
bm25_restricted_texto.vectorizer = vectorizer_restricted # same vocab
bm25_restricted_texto.fit(texto_documents)
embeddings_restricted_texto = bm25_restricted_texto.transform(texto_documents)

index_path_texto = Path(f'/work/sandbox/BM25_Filtered_Voc/indexes/texto')
with open(index_path_texto / 'documents.pkl', 'wb') as f:
    pickle.dump(texto_documents, f)
    
with open(index_path_texto / 'vectorizer.pkl', 'wb') as f:
    pickle.dump(bm25_restricted_texto, f)

scipy.sparse.save_npz(index_path_texto / 'embeddings.npz', embeddings_restricted_texto)

# Create a new SparseBM25Vectorizer instance for resumen documents
bm25_restricted_resumen = SparseBM25Vectorizer(k1=1.5, b=0.75)
bm25_restricted_resumen.vectorizer = vectorizer_restricted # same vocab
bm25_restricted_resumen.fit(resumen_documents)
embeddings_restricted_resumen = bm25_restricted_resumen.transform(resumen_documents)

index_path_resumen = Path(f'/work/sandbox/BM25_Filtered_Voc/indexes/resumen')
with open(index_path_resumen / 'documents.pkl', 'wb') as f:
    pickle.dump(resumen_documents, f)
    
with open(index_path_resumen / 'vectorizer.pkl', 'wb') as f:
    pickle.dump(bm25_restricted_resumen, f)

scipy.sparse.save_npz(index_path_resumen / 'embeddings.npz', embeddings_restricted_resumen)

### Paso 3: Recrear los índices con binary = True

In [11]:
import sys
from pathlib import Path
import pickle

# Add parent directory to Python path
# Assuming your project root is `/work`, add it to the system path
project_root = Path('/work')
sys.path.append(str(project_root))
from src.utils.data_utils import load_documents
from src.utils.index_classes import SparseBM25Vectorizer

from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse

# load documents
texto_doc_path = Path(f'/work/data/indexes/OEB_texto_BM25sparseindex')
resumen_doc_path = Path(f'/work/data/indexes/OEB_resumen_BM25sparseindex')

with open(texto_doc_path / "documents.pkl", "rb") as f:
    texto_documents = pickle.load(f)

with open(resumen_doc_path / "documents.pkl", "rb") as f:
    resumen_documents = pickle.load(f)

# Build the vectorizer with restricted vocabulary
vectorizer_restricted = CountVectorizer(vocabulary=allowed_vocab_dict, 
                                        binary=True, 
                                        lowercase=True, 
                                        token_pattern=r'\b\w+\b')

# Create a new SparseBM25Vectorizer instance for texto documents
bm25_restricted_texto = SparseBM25Vectorizer(k1=1.5, b=0.75)
bm25_restricted_texto.vectorizer = vectorizer_restricted # same vocab
bm25_restricted_texto.fit(texto_documents)
embeddings_restricted_texto = bm25_restricted_texto.transform(texto_documents)

index_path_texto = Path(f'/work/sandbox/BM25_Filtered_Voc/indexes_true/texto')
with open(index_path_texto / 'documents.pkl', 'wb') as f:
    pickle.dump(texto_documents, f)
    
with open(index_path_texto / 'vectorizer.pkl', 'wb') as f:
    pickle.dump(bm25_restricted_texto, f)

scipy.sparse.save_npz(index_path_texto / 'embeddings.npz', embeddings_restricted_texto)

# Create a new SparseBM25Vectorizer instance for resumen documents
bm25_restricted_resumen = SparseBM25Vectorizer(k1=1.5, b=0.75)
bm25_restricted_resumen.vectorizer = vectorizer_restricted # same vocab
bm25_restricted_resumen.fit(resumen_documents)
embeddings_restricted_resumen = bm25_restricted_resumen.transform(resumen_documents)

index_path_resumen = Path(f'/work/sandbox/BM25_Filtered_Voc/indexes_true/resumen')
with open(index_path_resumen / 'documents.pkl', 'wb') as f:
    pickle.dump(resumen_documents, f)
    
with open(index_path_resumen / 'vectorizer.pkl', 'wb') as f:
    pickle.dump(bm25_restricted_resumen, f)

scipy.sparse.save_npz(index_path_resumen / 'embeddings.npz', embeddings_restricted_resumen)